In [ ]:
# Standard libraries for handling in-memory bytes and zip archives
import io
import zipfile

# Pandas is used to load CSV data into DataFrames and merge them
import pandas as pd

# Requests is used to download the monthly Divvy zip files from the web
import requests

This notebook downloads Divvy bike trip data from July 2025 through June 2026, merges the monthly files into a single dataset, removes trailing metadata columns, and saves the cleaned result as a CSV file.


In [ ]:
# Build a list of Divvy trip data URLs covering July 2025 through June 2026.
# The source file names are formatted as YYYYMM-divvy-tripdata.zip.
urls = []
for year in range(2025, 2027):
    start = 7 if year == 2025 else 1
    end = 13 if year == 2025 else 7
    for i in range(start, end):
        # Ensure two-digit month formatting, e.g. '07' and '10'.
        num = "0" + str(i) if len(str(i)) == 1 else str(i)
        urls.append(f"https://divvy-tripdata.s3.amazonaws.com/{year}{num}-divvy-tripdata.zip")

# The `urls` list now contains 12 monthly zip file locations.

In [ ]:
# Download each monthly zip archive, extract the CSV file inside, and load it into a DataFrame.
# All monthly DataFrames are collected and then concatenated into a single dataset.

list_df_temp = []
for url in urls:
    response = requests.get(url)
    zip_bytes = io.BytesIO(response.content)
    
    with zipfile.ZipFile(zip_bytes) as z:
        for filename in z.namelist():
            # Only process CSV files inside the archive.
            if filename.endswith('.csv'):
                with z.open(filename) as f:
                    df_file = pd.read_csv(f, encoding='latin-1')
                    list_df_temp.append(df_file)

# Concatenate all monthly trip records into one DataFrame.
df = pd.concat(list_df_temp, ignore_index=True)
print(f"Finish! Total rows concatenated: {len(df)}")

Finish! Total rows concated: 5932350


In [ ]:
# Remove the last three columns from the dataset.
# These columns are assumed to be trailing metadata or extra fields not needed in the final export.

columns = df.columns[-3:].tolist()
df_final = df.drop(columns=columns)

In [ ]:
# Save the cleaned trip data to a CSV file in the current working directory.
df_final.to_csv("bike_trip_data(July 2025-June 2026).csv", index=False)
